<div style="width: 100%; clear: both;">
<div style="float: left; width: 50%;">
<img src="http://www.uoc.edu/portal/_resources/common/imatges/marca_UOC/UOC_Masterbrand.jpg", align="left">
</div>
</div>
<div style="float: right; width: 50%;">
<p style="margin: 0; padding-top: 22px; text-align:right;">M2.877 · Anàlisi de sentiments i textos</p>
<p style="margin: 0; text-align:right;">Màster universitari de Ciències de Dades (Data science)</p>
<p style="margin: 0; text-align:right; padding-button: 100px;">Estudis d'Informàtica, Multimèdia i Telecomunicacions</p>
</div>
</div>
<div style="width: 100%; clear: both;">
<div style="width:100%;">&nbsp;</div>

# Mòdul 5: Deep learning per a l'anàlisi de sentiments

## Anàlisi de sentiments

En aquest notebook, veurem dos exemples d'aplicació de models de deep learning per a l'anàlisi de sentiments:

- BiLSTM: LSTM bidireccionals
- Classificador a partir d'embeddings BERT

## PyTorch

Per als scripts d'aquest mòdul, farem servir el framework de deep learning PyTorch.

PyTorch és un framework relativament jove, comparat amb altres frameworks de deep learning. Malgrat això, la seva popularitat i la seva comunitat activa han crescut molt durant els dos últims anys, i en destaca especialment l'ús en tasques NLP.

Un dels motius pels quals és tan popular és la facilitat per debugar el codi que defineix les arquitectures del model.

## PASSOS PREVIS

Pytorch està implementat a la llibreria `torch`. A més, també farem servir `torchtext`, una extensió de `torch` per treballar amb dades textuals.
Per començar, carregarem torch i els mòduls necessaris de torchtext, i inicialitzarem l'entorn.

In [ ]:
import torch
from torchtext import data
from torchtext import datasets

SEED = 1234

torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# Obtenció de dades

Treballarem amb el dataset de yelp treballat al mòdul 3.
El dataset ara només conté el camp de text ('text') i el camp del sentiment ('sentiment') que s'ha obtingut a partir del camp stars en identificar amb un 0 les opinions amb una estrella i, amb un 1, les opinions amb cinc estrelles (vegeu notebook 3).

In [ ]:
import pandas as pd

df = pd.read_csv('data/yelp_binary.csv')

df.head()

,text,sentiment
0,My wife took me here on my birthday for breakf...,1
1,I have no idea why some people give bad review...,1
2,"Rosie, Dakota, and I LOVE Chaparral Dog Park!!...",1
3,General Manager Scott Petello is a good egg!!!...,1
4,Drop what you're doing and drive here. After I...,1


Per carregar el dataset per torch, necessitem definir primer els camps que farem servir i, després, crear el dataset com a dataset de torchtext.



El primer que hem de fer és definir els camps amb els quals treballarà el model: inputs i outputs. En aquest cas, tenim un camp TEXT per a l'entrada, que consistirà en el comentari que analitzarem, i un camp LABEL per a la sortida, que consistirà en l'etiqueta de la polaritat.
Inicialitzem el camp TEXT amb el tokenitzador de `spacy`. Si no s'especifica, el tokenitzador per defecte divideix el text a partir dels espais.


In [ ]:
TEXT = data.Field(tokenize = 'spacy', include_lengths = True)
LABEL = data.LabelField(dtype = torch.float)

A continuació, hem de definir els camps amb els quals treballarà el model: inputs i outputs. En aquest cas, tenim un camp TEXT per a l'entrada, que consistirà en el comentari que analitzarem, i un camp LABEL per a la sortida, que consistirà en l'etiqueta de la polaritat.
Inicialitzem el camp TEXT amb el tokenitzador de `spacy`. Si no s'especifica, el tokenitzador per defecte divideix el text a partir dels espais.

Ara, podrem carregar les dades en un dataset de `torchtext`. Per a això, definim una variable `fields` amb els camps definits en l'ordre en què es trobaran al dataset.

Aquesta sintaxi ens permet carregar camps concrets dels datasets. Si volem ometre un camp, afegim la tupla (None, None).

In [ ]:
#definir els camps
fields = [('text', TEXT), ('label', LABEL)]

#crear el dataset
train_data = data.TabularDataset(path = 'data/yelp_binary.csv',
                                        format = 'csv',
                                        fields = fields,
                                        skip_header = True)

Podem veure els exemples carregats en l'atribut `examples`; cadascun d'ells té accés als camps mitjançant el nom que els hem donat.

In [ ]:
print(train_data.examples[0].text)
print(train_data.examples[0].label)

['My', 'wife', 'took', 'me', 'here', 'on', 'my', 'birthday', 'for', 'breakfast', 'and', 'it', 'was', 'excellent', '.', ' ', 'The', 'weather', 'was', 'perfect', 'which', 'made', 'sitting', 'outside', 'overlooking', 'their', 'grounds', 'an', 'absolute', 'pleasure', '.', ' ', 'Our', 'waitress', 'was', 'excellent', 'and', 'our', 'food', 'arrived', 'quickly', 'on', 'the', 'semi', '-', 'busy', 'Saturday', 'morning', '.', ' ', 'It', 'looked', 'like', 'the', 'place', 'fills', 'up', 'pretty', 'quickly', 'so', 'the', 'earlier', 'you', 'get', 'here', 'the', 'better', '.', '\n\n', 'Do', 'yourself', 'a', 'favor', 'and', 'get', 'their', 'Bloody', 'Mary', '.', ' ', 'It', 'was', 'phenomenal', 'and', 'simply', 'the', 'best', 'I', "'ve", 'ever', 'had', '.', ' ', 'I', "'m", 'pretty', 'sure', 'they', 'only', 'use', 'ingredients', 'from', 'their', 'garden', 'and', 'blend', 'them', 'fresh', 'when', 'you', 'order', 'it', '.', ' ', 'It', 'was', 'amazing', '.', '\n\n', 'While', 'EVERYTHING', 'on', 'the', 'menu

# Obtenció de dades

Treballarem amb el dataset de yelp treballat al mòdul 3.
El dataset ara només conté el camp de text ('text') i el camp del sentiment ('sentiment') que s'ha obtingut a partir del camp stars en identificar amb un 0 les opinions amb una estrella i, amb un 1, les opinions amb cinc estrelles (vegeu notebook 3).

Ara, hem de construir els vocabularis dels camps, però abans dividirem el dataset en train, validation i test, per tenir en compte només el text en train.

In [ ]:
#dividim el dataset entre un conjunt d'entrenament i un de validació
import random

train_data, valid_data, test_data = train_data.split(split_ratio = [0.6, 0.2, 0.2], random_state = random.seed(SEED))

Per construir el vocabulari (look-up table) per representar els tokens com a vectors, carregarem els word embeddings preentrenats de GloVe. Recordem que per a l'anglès hi ha diversos models GloVe disponibles al web https://nlp.stanford.edu/projects/glove/. En aquest cas, carregarem els que tenen un vocabulari menys extens i que consten de cent dimensions. Des de torchtext, es poden carregar amb facilitat, encara que el procés pot trigar uns quants minuts.


In [ ]:
MAX_VOCAB_SIZE = 25_000

TEXT.build_vocab(train_data, 
                 max_size = MAX_VOCAB_SIZE, 
                 vectors = "glove.6B.100d", 
                 unk_init = torch.Tensor.normal_)


Construïm també un diccionari per a les etiquetes.

In [ ]:
LABEL.build_vocab(train_data)


Ara, podem explorar els exemples en detall, com hem vist abans, amb els seus respectius camps `text` i `label`:

In [ ]:
print(train_data.examples[0].text)
print(train_data.examples[0].label)

['Yummy', '!']
1


I comprovar com n'està, de balancejat, el conjunt de dades:

In [ ]:
LABEL.vocab.freqs

Counter({'1': 2005, '0': 447})

## PREPARACIÓ DE L'ENTRENAMENT

Un cop arribats a aquest punt, a part de definir algunes característiques de l'entrenament, com la quantitat d'exemples per batch, podem definir l'`iterator` que ens permetrà recórrer els exemples del conjunt d'entrenament en batches per entrenar el model. Farem servir el `BucketIterator`, la principal característica del qual és que agrupa els exemples segons la seva longitud, de manera que es minimitza en cada batch la quantitat de tokens per a padding que cal afegir per completar les entrades.

In [ ]:
BATCH_SIZE = 16

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_iterator, valid_iterator, test_iterator = data.BucketIterator.splits(
    (train_data, valid_data, test_data), 
    batch_size = BATCH_SIZE,
    sort_key = lambda x: len(x.text),
    sort_within_batch = False,
    device = device)


# MODEL 1:
# BiLSTM

## DEFINICIÓ DEL MODEL

El model proposat consisteix en:
- una capa embedding
- una capa BiLSTM, és a dir, LSTM bidireccional
- una capa fully-connected introduïda pel layer Linear (que equivaldria al `Dense` de Keras)



Per definir un model en `torch`, hem de definir els mètodes `__init__` i `forward`. En el primer, es defineixen els layers i, en el segon, l'execució del forward pass.

In [ ]:
import torch.nn as nn

class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, 
                 bidirectional, dropout, pad_idx):
        
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx = pad_idx)
        
        self.rnn = nn.LSTM(embedding_dim, 
                           hidden_dim, 
                           num_layers=n_layers, 
                           bidirectional=bidirectional, 
                           dropout=dropout)
        
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, text, text_lengths):
              
        with torch.no_grad():
            embedded = self.dropout(self.embedding(text))
                
        #pack sequence
        packed_embedded = nn.utils.rnn.pack_padded_sequence(embedded, text_lengths, enforce_sorted=False)
        
        packed_output, (hidden, cell) = self.rnn(packed_embedded)
        
        output, output_lengths = nn.utils.rnn.pad_packed_sequence(packed_output)
        
        hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim = 1))
                            
        return self.fc(hidden)

Inicialitzem el model.

In [ ]:
INPUT_DIM = len(TEXT.vocab)
EMBEDDING_DIM = 100
HIDDEN_DIM = 256
OUTPUT_DIM = 1
N_LAYERS = 2
BIDIRECTIONAL = True
DROPOUT = 0.5
PAD_IDX = TEXT.vocab.stoi[TEXT.pad_token]

model = BiLSTM(INPUT_DIM, 
            EMBEDDING_DIM, 
            HIDDEN_DIM, 
            OUTPUT_DIM, 
            N_LAYERS, 
            BIDIRECTIONAL, 
            DROPOUT, 
            PAD_IDX)

Podem comprovar el nombre de 
 que seran entrenats.

In [ ]:
#Nombre de paràmetres
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'El model té {count_parameters(model):,} paràmetres')

El model té 4,247,157 paràmetres


Copiem els word embeddings carregats en els pesos de l'embedding layer i afegim embeddings per als tokens per a padding i per a paraules fora del vocabulari, `<pad>` i `<unk>`, respectivament.

In [ ]:
pretrained_embeddings = TEXT.vocab.vectors
model.embedding.weight.data.copy_(pretrained_embeddings)

UNK_IDX = TEXT.vocab.stoi[TEXT.unk_token]

model.embedding.weight.data[UNK_IDX] = torch.zeros(EMBEDDING_DIM)
model.embedding.weight.data[PAD_IDX] = torch.zeros(EMBEDDING_DIM)

print(model.embedding.weight.data)


tensor([[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [-0.3398,  0.2094,  0.4635,  ..., -0.2339,  0.4730, -0.0288],
        ...,
        [ 0.5362, -2.6718, -0.8489,  ..., -0.8722, -0.5047, -0.5324],
        [-2.0057,  2.0427, -0.3378,  ..., -0.3376,  1.0165,  2.6028],
        [ 0.7556, -1.2552, -0.8194,  ..., -0.3970, -1.0552,  0.6198]])


Si volem congelar els paràmetres de la capa embedding, podem marcar requires_grad com a fals en els paràmetres en qüestió.


In [ ]:
for param in model.embedding.parameters():
    param.requires_grad = False

In [ ]:
#Nombre de paràmetres
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'El model té {count_parameters(model):,} paràmetres')

El model té 2,310,657 paràmetres


# PREPARACIÓ DE L'ENTRENAMENT

A continuació, definim l'`optimizer` i `loss` function.

In [ ]:
import torch.optim as optim

In [ ]:
optimizer = optim.Adam(model.parameters())
criterion = nn.BCEWithLogitsLoss() 

El criteri seleccionat (Binary Crossentropy with logits `BCEWithLogitsLoss`) ja inclou la funció d'activació sigmoide i la binary_crossentropy, però, a més, hem de definir una funció per a l'accuracy per tal de poder avaluar el model.

In [ ]:
def binary_accuracy(preds, y):
    rounded_preds = torch.round(torch.sigmoid(preds))
    corrections = (rounded_preds == y).float()
    acc = corrections.sum() / len(corrections)
    return acc

Passem les computacions a la GPU, si està disponible.

In [ ]:
criterion = criterion.to(device)
model = model.to(device)

Només ens falta definir els loops per entrenar i avaluar el model. Per a això, definim dues funcions molt semblants, `train` i `evaluate`. A evaluate, no s'actualitzen els pesos ni es calculen els gradients, i, a més, s'inhibeixen el dropout i la normalització del batch, que ara no es fa servir, però ens podria servir per a altres models.

In [ ]:
def train(model, iterator, optimizer, criterion):
    
    epoch_loss = 0
    epoch_acc = 0
    
    model.train() # habilita dropout i batch normalization
    
    for batch in iterator:
        optimizer.zero_grad()  
        text, text_lengths = batch.text
        predictions = model(text, text_lengths).squeeze(1)        
        loss = criterion(predictions, batch.label)        
        acc = binary_accuracy(predictions, batch.label)        
        loss.backward()     
        optimizer.step() # actualitza els pesos       
        epoch_loss += loss.item()
        epoch_acc += acc.item()
        
    return epoch_loss / len(iterator), epoch_acc / len(iterator)

def evaluate(model, iterator, criterion):
    
    epoch_loss = 0
    epoch_acc = 0
    
    model.eval()  # deshabilita dropout i batch normalization
    
    with torch.no_grad(): # per no calcular els gradients durant les computacions
    
        for batch in iterator:
            text, text_lengths = batch.text
            predictions = model(text, text_lengths).squeeze(1)           
            loss = criterion(predictions, batch.label)
            acc = binary_accuracy(predictions, batch.label)
            epoch_loss += loss.item()
            epoch_acc += acc.item()
        
    return epoch_loss / len(iterator), epoch_acc / len(iterator)

#Funció de logs per als temps de les epochs

import time

def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

# ENTRENAMENT DEL MODEL

In [ ]:
N_EPOCHS = 5

best_valid_loss = float('inf')

for epoch in range(N_EPOCHS):

    start_time = time.time()
    
    train_loss, train_acc = train(model, train_iterator, optimizer, criterion)
    valid_loss, valid_acc = evaluate(model, valid_iterator, criterion)
    
    end_time = time.time()

    epoch_mins, epoch_secs = epoch_time(start_time, end_time)
    
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'model-bilstm1.pt')
    
    print(f'Epoch: {epoch+1:02} | Epoch Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc*100:.2f}%')

Epoch: 01 | Epoch Time: 0m 17s
	Train Loss: 0.475 | Train Acc: 81.37%
	 Val. Loss: 0.446 |  Val. Acc: 81.25%
Epoch: 02 | Epoch Time: 0m 16s
	Train Loss: 0.450 | Train Acc: 81.98%
	 Val. Loss: 0.501 |  Val. Acc: 75.12%
Epoch: 03 | Epoch Time: 0m 17s
	Train Loss: 0.451 | Train Acc: 81.25%
	 Val. Loss: 0.462 |  Val. Acc: 81.25%
Epoch: 04 | Epoch Time: 0m 16s
	Train Loss: 0.450 | Train Acc: 81.86%
	 Val. Loss: 0.467 |  Val. Acc: 80.89%
Epoch: 05 | Epoch Time: 0m 17s
	Train Loss: 0.441 | Train Acc: 82.10%
	 Val. Loss: 0.439 |  Val. Acc: 81.97%


Tornem a definir ara el mateix model, i deixem que els paràmetres de la capa embedding també s'entrenin.

In [ ]:
model = BiLSTM(INPUT_DIM, 
            EMBEDDING_DIM, 
            HIDDEN_DIM, 
            OUTPUT_DIM, 
            N_LAYERS, 
            BIDIRECTIONAL, 
            DROPOUT, 
            PAD_IDX)

In [ ]:
pretrained_embeddings = TEXT.vocab.vectors
model.embedding.weight.data.copy_(pretrained_embeddings)

UNK_IDX = TEXT.vocab.stoi[TEXT.unk_token]

model.embedding.weight.data[UNK_IDX] = torch.zeros(EMBEDDING_DIM)
model.embedding.weight.data[PAD_IDX] = torch.zeros(EMBEDDING_DIM)

print(model.embedding.weight.data)


tensor([[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [-0.3398,  0.2094,  0.4635,  ..., -0.2339,  0.4730, -0.0288],
        ...,
        [ 0.5362, -2.6718, -0.8489,  ..., -0.8722, -0.5047, -0.5324],
        [-2.0057,  2.0427, -0.3378,  ..., -0.3376,  1.0165,  2.6028],
        [ 0.7556, -1.2552, -0.8194,  ..., -0.3970, -1.0552,  0.6198]])


In [ ]:
model = model.to(device)
optimizer = optim.Adam(model.parameters())


In [ ]:
N_EPOCHS = 5

best_valid_loss = float('inf')

for epoch in range(N_EPOCHS):

    start_time = time.time()
    
    train_loss, train_acc = train(model, train_iterator, optimizer, criterion)
    valid_loss, valid_acc = evaluate(model, valid_iterator, criterion)
    
    end_time = time.time()

    epoch_mins, epoch_secs = epoch_time(start_time, end_time)
    
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'model-bilstm2.pt')
    
    print(f'Epoch: {epoch+1:02} | Epoch Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc*100:.2f}%')

Epoch: 01 | Epoch Time: 0m 17s
	Train Loss: 0.469 | Train Acc: 81.78%
	 Val. Loss: 0.479 |  Val. Acc: 81.01%
Epoch: 02 | Epoch Time: 0m 16s
	Train Loss: 0.454 | Train Acc: 81.62%
	 Val. Loss: 0.459 |  Val. Acc: 81.01%
Epoch: 03 | Epoch Time: 0m 16s
	Train Loss: 0.425 | Train Acc: 82.35%
	 Val. Loss: 0.437 |  Val. Acc: 81.25%
Epoch: 04 | Epoch Time: 0m 16s
	Train Loss: 0.431 | Train Acc: 82.43%
	 Val. Loss: 0.435 |  Val. Acc: 81.01%
Epoch: 05 | Epoch Time: 0m 17s
	Train Loss: 0.413 | Train Acc: 82.39%
	 Val. Loss: 0.425 |  Val. Acc: 82.09%


# TEST: Avaluació del model

Per carregar els models entrenats, primer, cal inicialitzar un model amb la mateixa arquitectura.

In [ ]:
model = BiLSTM(INPUT_DIM, 
            EMBEDDING_DIM, 
            HIDDEN_DIM, 
            OUTPUT_DIM, 
            N_LAYERS, 
            BIDIRECTIONAL, 
            DROPOUT, 
            PAD_IDX)

model = model.to(device)

pretrained_embeddings = TEXT.vocab.vectors
model.embedding.weight.data.copy_(pretrained_embeddings)

UNK_IDX = TEXT.vocab.stoi[TEXT.unk_token]

model.embedding.weight.data[UNK_IDX] = torch.zeros(EMBEDDING_DIM)
model.embedding.weight.data[PAD_IDX] = torch.zeros(EMBEDDING_DIM)



I, després, carregar-lo amb els seus paràmetres mitjançant l'ordre:
`model.load_state_dict(torch.load(nom_del_fitxer))`.

In [ ]:
model.load_state_dict(torch.load('model-bilstm1.pt'))

test_loss, test_acc = evaluate(model, test_iterator, criterion)

print(f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}%')


model.load_state_dict(torch.load('model-bilstm2.pt'))

test_loss, test_acc = evaluate(model, test_iterator, criterion)

print(f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}%')


Test Loss: 0.414 | Test Acc: 81.97%
Test Loss: 0.406 | Test Acc: 82.45%


# MODEL 2:

# MODEL A PARTIR D'EMBEDDINGS BERT

Primer, carregarem el model i el tokenizer per a BERT.

In [ ]:
from transformers import BertModel, BertTokenizer

I els definim.

Perquè el tokenizer funcioni correctament, ha de correspondre a l'entrenat amb el mateix model BERT que s'utilitzarà; en aquest cas, farem servir 'bert-base-uncased'.

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert = BertModel.from_pretrained('bert-base-uncased')

## Preparació de dades per a BERT



Les representacions que el model BERT que hem carregat pot processar i, per a les quals obté representacions vectorials, tenen un límit de tokens; el límit, en aquest cas, és 512, per la qual cosa, hem de limitar les seqüències a aquest valor.

Ho podem comprovar amb el mateix tokenizer amb l'ordre següent:

In [ ]:
tokenizer.max_model_input_sizes['bert-base-uncased']

512

En aquest cas, en comptes de crear una look up table, afegirem a la definició del field TEXT el pipeline per processar-lo i obtenir la llista d'índexs per ser utilitzats com a input de BERT.

In [ ]:
def tokenize_and_cut(sentence):
    maxlen = tokenizer.max_model_input_sizes['bert-base-uncased']
    tokens = tokenizer.tokenize(sentence) 
    tokens = tokens[:maxlen-2]
    return tokens

from torchtext import data

TEXT = data.Field(batch_first = True,
                  use_vocab = False,
                  tokenize = tokenize_and_cut,
                  preprocessing = tokenizer.convert_tokens_to_ids,
                  init_token = tokenizer.cls_token_id,
                  eos_token = tokenizer.sep_token_id,
                  pad_token = tokenizer.pad_token_id,
                  unk_token = tokenizer.unk_token_id)

LABEL = data.LabelField(dtype = torch.float)

Tornem a carregar el dataset amb els nous fields definits.

In [ ]:
fields = [('text', TEXT), ('label', LABEL)]

train_data = data.TabularDataset(path = 'data/yelp_binary.csv',
                                        format = 'csv',
                                        fields = fields,
                                        skip_header = True)

train_data, valid_data, test_data = train_data.split(split_ratio = [0.6, 0.2, 0.2], random_state = random.seed(SEED))

In [ ]:
print(vars(train_data.examples[0]))

{'text': [9805, 18879, 999], 'label': '1'}


In [ ]:
LABEL.build_vocab(train_data)
LABEL.vocab.freqs

Counter({'1': 2005, '0': 447})

Creem l'`iterator`i preparem l'entrenament.

In [ ]:
BATCH_SIZE = 64

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_iterator, valid_iterator, test_iterator = data.BucketIterator.splits(
    (train_data, valid_data, test_data), 
    batch_size = BATCH_SIZE, 
    sort_key = lambda x: len(x.text),
    sort_within_batch = False,
    device = device)

# Definició del model

El model és molt semblant a l'anterior, però, en aquest cas, farem servir embeddings de BERT. Per a això, definirem tres capes:
- BERT: afegirem una primera capa amb el model BERT carregat (que consisteix, en si mateix, en una arquitectura complexa). 
- GRU unit: després, farem servir els embeddings de cada token com a input d'una gated recurrent unit, també bidireccional. És a dir, en aquest cas, farem servir els embeddings de cada token, en comptes d'utilitzar el de tota la frase, és a dir, en lloc de fer servir el del token `[CLS]`.
- I de nou, una capa fully-connected per a la classificació.

Els índexs de segment són opcionals en aquest cas, ja que fem servir una sola frase d'entrada, i no els farem servir.

In [ ]:
import torch.nn as nn

class BERTGRUSentiment(nn.Module):
    def __init__(self,
                 bert,
                 hidden_dim,
                 output_dim,
                 n_layers,
                 bidirectional,
                 dropout):
        
        super().__init__()
        
        self.bert = bert
        
        embedding_dim = bert.config.to_dict()['hidden_size']
        
        self.rnn = nn.GRU(embedding_dim,
                          hidden_dim,
                          num_layers = n_layers,
                          bidirectional = bidirectional,
                          batch_first = True,
                          dropout = 0 if n_layers < 2 else dropout)
        
        self.out = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, text):
        
        with torch.no_grad(): # així, congelem els paràmetres de BERT durant l'entrenament
            embedded = bert(text)[0]
                        
        _, hidden = self.rnn(embedded)
                
        if self.rnn.bidirectional:
            hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim = 1))
        else:
            hidden = self.dropout(hidden[-1,:,:])
                        
        output = self.out(hidden)
                
        return output

Creem la instància del model.

In [ ]:
HIDDEN_DIM = 256
OUTPUT_DIM = 1
N_LAYERS = 2
BIDIRECTIONAL = True
DROPOUT = 0.25

model = BERTGRUSentiment(bert,
                         HIDDEN_DIM,
                         OUTPUT_DIM,
                         N_LAYERS,
                         BIDIRECTIONAL,
                         DROPOUT)

Comptem de nou el nombre de paràmetres que cal entrenar.

In [ ]:
#Nombre de paràmetres
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'El model té {count_parameters(model):,} paràmetres')

El model té 112,241,409 paràmetres


Com veiem, són molts més que abans, ja que inclouen tots els paràmetres del model BERT. En aquest cas, el que farem és marcar en el forward pass que no volem que s'entreni la part de bert. Ho fem amb l'scope `with torch.no_grad ():`


# Preparació de l'entrenament

Per entrenar el model, continuarem fent servir el mateix `criterion` i `optimizer`.

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters())

model = model.to(device)
criterion = criterion.to(device)

Tornem a redefinir les funcions `train` i `evaluate`, ja que, de nou, l'input del model és només el camp text (i no la longitud, que ara serà fixa).

In [ ]:
def train(model, iterator, optimizer, criterion):
    
    epoch_loss = 0
    epoch_acc = 0
    
    model.train()
    
    for batch in iterator:
        
        optimizer.zero_grad()
        
        predictions = model(batch.text).squeeze(1)
        
        loss = criterion(predictions, batch.label)
        
        acc = binary_accuracy(predictions, batch.label)
        
        loss.backward()
        
        optimizer.step()
        
        epoch_loss += loss.item()
        epoch_acc += acc.item()
        
    return epoch_loss / len(iterator), epoch_acc / len(iterator)

def evaluate(model, iterator, criterion):
    
    epoch_loss = 0
    epoch_acc = 0
    
    model.eval()
    
    with torch.no_grad():
    
        for batch in iterator:

            predictions = model(batch.text).squeeze(1)
            
            loss = criterion(predictions, batch.label)
            
            acc = binary_accuracy(predictions, batch.label)

            epoch_loss += loss.item()
            epoch_acc += acc.item()
        
    return epoch_loss / len(iterator), epoch_acc / len(iterator)

# Entrenament

In [ ]:
import time
N_EPOCHS = 5

best_valid_loss = float('inf')

for epoch in range(N_EPOCHS):
    
    start_time = time.time()
    
    train_loss, train_acc = train(model, train_iterator, optimizer, criterion)
    valid_loss, valid_acc = evaluate(model, valid_iterator, criterion)
        
    end_time = time.time()
        
    epoch_mins, epoch_secs = epoch_time(start_time, end_time)
        
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'tut6-model.pt')
    
    print(f'Epoch: {epoch+1:02} | Epoch Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc*100:.2f}%')


Epoch: 01 | Epoch Time: 2m 49s
	Train Loss: 0.261 | Train Acc: 89.74%
	 Val. Loss: 0.186 |  Val. Acc: 92.89%
Epoch: 02 | Epoch Time: 2m 45s
	Train Loss: 0.174 | Train Acc: 93.29%
	 Val. Loss: 0.092 |  Val. Acc: 95.88%
Epoch: 03 | Epoch Time: 2m 44s
	Train Loss: 0.118 | Train Acc: 95.58%
	 Val. Loss: 0.125 |  Val. Acc: 95.32%
Epoch: 04 | Epoch Time: 2m 48s
	Train Loss: 0.116 | Train Acc: 95.42%
	 Val. Loss: 0.086 |  Val. Acc: 96.32%
Epoch: 05 | Epoch Time: 2m 45s
	Train Loss: 0.065 | Train Acc: 97.64%
	 Val. Loss: 0.078 |  Val. Acc: 96.72%


# TEST

## Comparació amb model clàssic

A més, per comparar els resultats amb el model treballat al mòdul 3, calculem el recall, precision i F1 sobre el conjunt del test. Per a això, definim la funció següent:

In [ ]:
#modifiquem la funció perquè retorni les prediccions
def evaluate(model, iterator, criterion):
    predictions_all = []
    labels_all = []
    
    model.eval()  # deshabilita dropout i batch normalization
    
    with torch.no_grad(): # per no calcular els gradients durant les computacions
    
        for batch in iterator:
            predictions = model(batch.text).squeeze(1) 
            predictions_all +=  torch.round(torch.sigmoid(predictions)).flatten().cpu().numpy().tolist()
            labels_all += batch.label.flatten().cpu().numpy().tolist()
        
    return predictions_all, labels_all

predictions, labels = evaluate(model, test_iterator, criterion)


In [ ]:
from sklearn import metrics

#metrics.classification_report(labels, predictions, labels = [0.0,1.0])


Models clàssics:

|val | precision | recall | F1-score | support |
|----|----|-------|---- | ------ |
| 0 | 0.98 | 0.41 | 0.57 | 148 |
| 1 | 0.88 | 1.00 | 0.94 | 670 |

Model BiLSTM:

|val | precision | recall | F1-score | support |
|----|----|-------|---- | ----- |
| 0 | 0.87 | 0.32 | 0.47 | 143 |
| 1 | 0.87 | 0.99 | 0.93 |  674 |

Model BERT-GRU:

|val | precision | recall | F1-score | support |
|----|----|-------|---- | ----- |
| 0 | 0.95 | 0.89 | 0.92 | 143 |
| 1 | 0.98 | 0.99 | 0.98 |  674 |



